In [3]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD

In [4]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

print(movies.shape, ratings.shape)
ratings.head()

(9742, 3) (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

In [6]:
user_item_matrix = train.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

user_item_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,191005,193565,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

In [10]:
def recommend_user_based(user_id, top_n=5):
    # Get similarity scores for the user
    similar_users = user_similarity_df.loc[user_id]

    # Remove the user itself
    similar_users = similar_users.drop(user_id)

    # Align ratings matrix with similar users
    similar_users_ratings = user_item_matrix.loc[similar_users.index]

    # Weighted sum of ratings
    weighted_ratings = np.dot(similar_users.values, similar_users_ratings.values)

    # Normalize by sum of similarities
    scores = weighted_ratings / similar_users.sum()

    # Convert to Series
    scores = pd.Series(scores, index=user_item_matrix.columns)

    # Remove already watched movies
    watched = train[train['userId'] == user_id]['movieId']
    scores = scores.drop(watched, errors='ignore')

    # Top N recommendations
    top_movies = scores.sort_values(ascending=False).head(top_n).index

    return movies[movies['movieId'].isin(top_movies)][['title']]

In [11]:
recommend_user_based(user_id=1, top_n=5)

,title
277,"Shawshank Redemption, The (1994)"
507,Terminator 2: Judgment Day (1991)
659,"Godfather, The (1972)"
900,Raiders of the Lost Ark (Indiana Jones and the...
3638,"Lord of the Rings: The Fellowship of the Ring,..."


In [12]:
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

In [13]:
def recommend_item_based(user_id, top_n=5):
    user_ratings = user_item_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index

    scores = item_similarity_df[rated_items].dot(user_ratings[rated_items])
    scores = scores / item_similarity_df[rated_items].sum(axis=1)

    scores = scores.drop(rated_items, errors='ignore')
    top_movies = scores.sort_values(ascending=False).head(top_n).index

    return movies[movies['movieId'].isin(top_movies)][['title']]

In [14]:
recommend_item_based(user_id=1, top_n=5)

,title
1302,3 Ninjas: High Noon On Mega Mountain (1998)
8828,The Jinx: The Life and Deaths of Robert Durst ...
9277,Bakuman (2015)
9285,Gintama: The Final Chapter - Be Forever Yorozu...
9740,Bungo Stray Dogs: Dead Apple (2018)


In [15]:
def precision_at_k(model_func, k=5):
    precisions = []

    for user in test['userId'].unique():
        true_movies = test[test['userId'] == user]['movieId'].values
        if len(true_movies) == 0:
            continue

        try:
            recs = model_func(user, k)
            rec_ids = movies[movies['title'].isin(recs['title'])]['movieId'].values
        except:
            continue

        precision = len(set(rec_ids) & set(true_movies)) / k
        precisions.append(precision)

    return np.mean(precisions)

In [16]:
print("User-Based Precision@5:", precision_at_k(recommend_user_based, 5))
print("Item-Based Precision@5:", precision_at_k(recommend_item_based, 5))

User-Based Precision@5: 0.2665573770491803
Item-Based Precision@5: 0.0


In [17]:
svd = TruncatedSVD(n_components=50, random_state=42)
latent_matrix = svd.fit_transform(user_item_matrix)

In [18]:
svd_preds = np.dot(latent_matrix, svd.components_)
svd_preds_df = pd.DataFrame(
    svd_preds,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

In [19]:
def recommend_svd(user_id, top_n=5):
    scores = svd_preds_df.loc[user_id]
    watched = train[train['userId'] == user_id]['movieId']
    scores = scores.drop(watched, errors='ignore')

    top_movies = scores.sort_values(ascending=False).head(top_n).index
    return movies[movies['movieId'].isin(top_movies)][['title']]

In [20]:
recommend_svd(user_id=1, top_n=5)

,title
507,Terminator 2: Judgment Day (1991)
793,Die Hard (1988)
900,Raiders of the Lost Ark (Indiana Jones and the...
1576,Indiana Jones and the Temple of Doom (1984)
2078,"Sixth Sense, The (1999)"


In [21]:
print("SVD Precision@5:", precision_at_k(recommend_svd, 5))

SVD Precision@5: 0.3163934426229508
